# Indonesian Hybrid Dataset EDA

Exploratory analysis of the 5K+ real Indonesia profile-job pair dataset
generated from 507 seed records with skill-subset augmentation.

In [ ]:
import json
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
DATA_PATH = Path('../services/sbert/training/data/indonesian_profile_job_pairs_hybrid.jsonl')
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

## Basic Statistics

In [ ]:
print(df['pair_kind'].value_counts())
print()
print(df['split'].value_counts())
print()
print(f"Unique profiles: {df['profile_id'].nunique()}")
print(f"Unique jobs: {df['job_id'].nunique()}")

## Program Studi Distribution

In [ ]:
# Extract program studi from profile_text (first word(s) before target_role)
def extract_program(profile_text):
    parts = profile_text.split()
    # Program studi is typically 1-3 words before the target role
    if len(parts) >= 2:
        return parts[0]
    return 'Unknown'

positive_df = df[df['pair_kind'] == 'positive'].copy()
positive_df['program_studi'] = positive_df['profile_text'].apply(extract_program)

program_counts = positive_df['program_studi'].value_counts().head(20)
program_counts.plot(kind='barh', color='steelblue')
plt.title('Top 20 Program Studi Distribution (Positive Pairs)')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

## Skill Analysis

In [ ]:
all_job_skills = [skill for skills in df['job_skills'] for skill in skills]
skill_counts = Counter(all_job_skills).most_common(30)

skills, counts = zip(*skill_counts)
plt.barh(range(len(skills)), counts, color='coral')
plt.yticks(range(len(skills)), skills)
plt.title('Top 30 Job Skills in Dataset')
plt.xlabel('Frequency')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Matched Skills Analysis

In [ ]:
df['matched_skills_count'] = df['matched_skills'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

positive_df = df[df['pair_kind'] == 'positive']
negative_df = df[df['pair_kind'] == 'negative']

positive_df['matched_skills_count'].hist(bins=range(0, 10), ax=axes[0], alpha=0.7, color='green')
axes[0].set_title('Positive Pairs: Matched Skills Distribution')
axes[0].set_xlabel('Number of Matched Skills')
axes[0].set_ylabel('Count')

negative_df['matched_skills_count'].hist(bins=range(0, 10), ax=axes[1], alpha=0.7, color='red')
axes[1].set_title('Negative Pairs: Matched Skills Distribution')
axes[1].set_xlabel('Number of Matched Skills')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Positive pairs - mean matched skills: {positive_df['matched_skills_count'].mean():.2f}")
print(f"Negative pairs - mean matched skills: {negative_df['matched_skills_count'].mean():.2f}")

## Split Distribution by Pair Kind

In [ ]:
split_kind = df.groupby(['split', 'pair_kind']).size().unstack(fill_value=0)
split_kind.plot(kind='bar', stacked=True, color=['coral', 'steelblue'])
plt.title('Distribution by Split and Pair Kind')
plt.xlabel('Split')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Pair Kind')
plt.tight_layout()
plt.show()

## Dataset Summary

In [ ]:
summary = {
    'total_records': len(df),
    'positive_pairs': len(df[df['pair_kind'] == 'positive']),
    'negative_pairs': len(df[df['pair_kind'] == 'negative']),
    'unique_profiles': df['profile_id'].nunique(),
    'unique_jobs': df['job_id'].nunique(),
    'train_records': len(df[df['split'] == 'train']),
    'validation_records': len(df[df['split'] == 'validation']),
    'test_records': len(df[df['split'] == 'test']),
    'with_hard_negative': len(df[(df['pair_kind'] == 'positive') & (df['hard_negative_job_id'].notna())]),
    'avg_matched_skills_positive': positive_df['matched_skills_count'].mean(),
    'avg_matched_skills_negative': negative_df['matched_skills_count'].mean(),
}

for key, value in summary.items():
    print(f"{key}: {value:.2f}" if isinstance(value, float) else f"{key}: {value}")